## <b> <span style='color:#2ae4f5'>|</span> Analyzing Previous Marketing Campaign Patterns to Enhance Future Success  </b> 

## <b>1 <span style='color:#2ae4f5'>|</span> Introduction </b> 

<div style="color:white;display:fill;border-radius:8px;
            background-color:#03112A;font-size:150%;
            letter-spacing:1.0px;background-image: url(https://i.imgur.com/GVd0La1.png)">
    <p style="padding: 8px;color:white;"><b><b><span style='color:#2ae4f5''>1.1 |</span></b> Analyzing Previous Marketing Campaign Patterns to Enhance Future Success </b></p>
</div>
      
To enhance the efficacy of forthcoming **marketing campaigns** for a **financial institution**, a comprehensive analysis of the patterns exhibited in previous **marketing campaigns** is imperative. Through this analysis, valuable insights can be gained, enabling the identification of optimal strategies to implement for attaining significant advancements in **future campaigns**. By leveraging the knowledge acquired from studying past patterns, the financial institution can strive towards achieving heightened levels of success and effectiveness in its marketing endeavors.

## <b>2 <span style='color:#2ae4f5'>|</span> Import Libraries </b> 

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style= "darkgrid", color_codes = True)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve 
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings('ignore')

## <b> 3 <span style='color:#2ae4f5'>|</span> Import DataSet </b> 

In [ ]:
# Reading Dataset
train_data = pd.read_csv('/kaggle/input/bank-marketing-dataset/bank_customers_train.csv')
train_data. head()

In [ ]:
# More details about dataset
train_data.info()

## <b> 4 <span style='color:#2ae4f5'>|</span> Preprocessing And Feature Engineering </b> 

In [ ]:
# Binary Conversion of Categorical Variable in Training Data
train_data = train_data.replace({'y': {'yes': 1, 'no': 0}})

In [ ]:
# Visualization of Class Distribution in the Training Data
sns.countplot(x='y', data=train_data, palette = 'hls')
plt.show()

**As evident from the dataset, there is a significant class imbalance with an inadequate number of positive samples. To ensure optimal model training, I opted to balance the sample distribution by selecting a representative sample of both positive and negative samples**

In [ ]:
# Creating a Balanced Dataset for Binary Classification
subscribed = train_data[train_data['y'] == 1].sample(n= 4415)
non_subscribed = train_data[train_data['y'] == 0].sample(n= 7000)
new_dataset = pd.concat([subscribed, non_subscribed], axis=0)
new_dataset.shape

In [ ]:
# Missing Value Analysis in the New Dataset
new_dataset.isna().sum()

In [ ]:
# Dataset Information for the New Dataset
new_dataset.info()
sns.countplot(x='y', data=train_data, palette = 'hls')
plt.show()

In [ ]:
# Descriptive Statistics of the New Dataset
new_dataset.describe()

In [ ]:
# Visualization of Relationships between Variables
class RelationshipPlotter:
    
    def __init__(self, data):
        self.data = data
    
    def plot(self, x, y, plot_type):
        if plot_type == 'boxplot':
            sns.boxplot(data=self.data, x=x, y=y)
        elif plot_type == 'barplot':
            sns.barplot(data=self.data, x=x, y=y)
        plt.title(f"{x.capitalize()} vs. {y.capitalize()}")
        plt.show()

plotter = RelationshipPlotter(data=train_data)

# Plot the relationships between 
plotter.plot(x='y', y='age', plot_type='boxplot')
plotter.plot(x='y', y='duration', plot_type='boxplot')
plotter.plot(x='y', y='campaign', plot_type='boxplot')
plotter.plot(x='y', y='pdays', plot_type='barplot')
plotter.plot(x='y', y='previous', plot_type='boxplot')
plotter.plot(x='day_of_week', y='y', plot_type='barplot')
plotter.plot(x='day_of_week', y='y', plot_type='barplot')
plotter.plot(x='month', y='y', plot_type='barplot')
plotter.plot(x='job', y='y', plot_type='barplot')
plotter.plot(x='marital', y='y', plot_type='barplot')
plotter.plot(x='education', y='y', plot_type='barplot')
plotter.plot(x='default', y='y', plot_type='barplot')
plotter.plot(x='housing', y='y', plot_type='barplot')
plotter.plot(x='loan', y='y', plot_type='barplot')
plotter.plot(x='contact', y='y', plot_type='barplot')
plotter.plot(x='poutcome', y='y', plot_type='barplot')


In [ ]:
#Split dataset
x= new_dataset.drop(columns='y')
y= new_dataset['y']
X_train, X_val, y_train, y_val=train_test_split(x,y, shuffle=True, random_state=12, test_size=0.1)


In [ ]:
# The most frequent value for each categorical feature, considering whether y is negative or positive
list_features_isna=['job','marital','education','default','housing','loan']

def most_frequent(df, cols, target):
    list_of_most_frequent={}
    for col in list_features_isna:
        f = df.groupby(target)[col].apply(lambda x: x.mode().iloc[0])
        list_of_most_frequent[col]= f
    return pd.DataFrame(list_of_most_frequent)

most_frequent(X_train, list_features_isna, y)
most_frequent(X_val, list_features_isna, y)

In [ ]:
#Handling Missing Values in Categorical Features
list_features_isna=['job','marital','education','default','housing','loan']

def manage_missvalues(df,cols):
    
    for col in cols:
        df[col].fillna(df[col].mode()[0], inplace=True)

    return df

X_train = manage_missvalues(X_train, list_features_isna)
X_val = manage_missvalues(X_val, list_features_isna)


In [ ]:
#converting categorical features to int
def encode_binary_cate(df):
    dic_={
        'yes': 1,
        'no': 0   
    }
    
    for col in ['default','housing','loan']:
        df[col]=df[col].map(dic_)
    return df

X_train = encode_binary_cate(X_train)
X_val = encode_binary_cate(X_val)


# Encoding Categorical Features
def encode_categorical_features(df):

    for col in ['job','marital','education','contact','month','day_of_week','poutcome']:
        dummies = pd.get_dummies(df[col], dtype=int)
        df = pd.concat([df, dummies], axis=1)
        df = df.drop(labels=col, axis=1)
        
    return df

# Encode the categorical features in the training and test sets
X_train = encode_categorical_features(X_train)
X_val = encode_categorical_features(X_val)


In [ ]:
# Calculate the correlation matrix
corr_matrix = X_train.corr()
fig, ax = plt.subplots(figsize=(30, 20))
sns.heatmap(corr_matrix, cmap='coolwarm', annot=True, ax=ax)
ax.set_title('Correlation Matrix')
plt.show()

In [ ]:
# Scale the dataset using StandardScaler
scaler = StandardScaler()

X_V = X_val.values
scaled_x_train = scaler.fit_transform(X_train)
scaled_x_val = scaler.transform(X_V)

## <b> 5 <span style='color:#2ae4f5'>|</span> Using GridSearch to find optimal hyperparameters </b> 

In [ ]:
# searching best value of hyperparameters for training models by GridSearchCV
# Define a list of classifiers to use
classifiers = [
    LogisticRegression(),
    DecisionTreeClassifier(),
    RandomForestClassifier(),
    AdaBoostClassifier(),
    GradientBoostingClassifier(),
    KNeighborsClassifier()
]


# Define a parameter grid for each classifier
param_grids = [
    {'penalty': ['l1', 'l2'], 'C': [0.1, 1, 10]},
    {'max_depth': [3, 4, 5], 'min_samples_split': [2, 5, 10]},
    {'max_depth': [3, 4, 5], 'min_samples_split': [2, 5, 10], 'max_features': [2, 3, 4]},
    {'n_estimators': [50, 100, 200], 'learning_rate': [0.1, 0.5, 1]},
    {'max_depth': [3, 4, 5], 'min_samples_split': [2, 5, 10], 'n_estimators': [50, 100, 200], 'learning_rate': [0.1, 0.5, 1]},
    {'n_neighbors': [3, 5, 7], 'weights': ['uniform', 'distance'], 'p': [1, 2]}
]

# Perform a grid search for each classifier
for i, clf in enumerate(classifiers):
    grid_search = GridSearchCV(clf, param_grid=param_grids[i], scoring='accuracy', cv=5)
    grid_search.fit(scaled_x_train, y_train)
    print(f"Best parameters for classifier {i}: {grid_search.best_params_}")
    print(f"Training accuracy for classifier {i}: {grid_search.best_score_}")
    print(f"Test accuracy for classifier {i}: {grid_search.score(scaled_x_val, y_val)}")
    


## <b> 6 <span style='color:#2ae4f5'>|</span> Train and Evaluate Models</b> 

In [ ]:
# Define a list of classifiers to use
classifiers = [
    LogisticRegression(penalty='l2', C=1, random_state=42),
    DecisionTreeClassifier(max_depth=5, min_samples_split=2, random_state=42),
    RandomForestClassifier(n_estimators=100, max_depth=5, min_samples_split=2, max_features=4, random_state=42),
    SVC(kernel='rbf', C=0.1, gamma=0.01, random_state=42, probability=True),
    AdaBoostClassifier(base_estimator=DecisionTreeClassifier(max_depth=5, min_samples_split=2, random_state=42), n_estimators=200, learning_rate=0.5, random_state=42),
    GradientBoostingClassifier(n_estimators=100, max_depth=5, min_samples_split=5, learning_rate=0.1, random_state=42),
    KNeighborsClassifier(n_neighbors=7, weights='uniform', p=1)
]

In [ ]:
# Initialize empty lists to store the evaluation metrics
classifier_names = []
accuracies = []
precisions = []
recalls = []
f1_scores = []

# Loop through each classifier and fit it to the training data
for classifier in classifiers:
    classifier.fit(scaled_x_train, y_train)
    y_pred = classifier.predict(scaled_x_val)
    
    # Get the name of the classifier
    classifier_name = type(classifier).__name__
    
    # Calculate the evaluation metrics
    accuracy = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    
    # Append the evaluation metrics to the lists
    classifier_names.append(classifier_name)
    accuracies.append(accuracy)
    precisions.append(precision)
    recalls.append(recall)
    f1_scores.append(f1)

# Create a bar plot that shows all the evaluation metrics for each classifier
x = np.arange(len(classifier_names))
width = 0.2
sns.set_style('darkgrid')
fig, ax = plt.subplots(figsize=(15, 10))
rects1 = ax.bar(x - 1.5*width, accuracies, width, label='Accuracy')
rects2 = ax.bar(x - 0.5*width, precisions, width, label='Precision')
rects3 = ax.bar(x + 0.5*width, recalls, width, label='Recall')
rects4 = ax.bar(x + 1.5*width, f1_scores, width, label='F1 Score')

# Add some text for labels, title and legend
ax.set_ylabel('Score')
ax.set_title('Evaluation Metrics for Classifiers')
ax.set_xticks(x)
ax.set_xticklabels(classifier_names, rotation=45, ha="right")
ax.legend(loc='lower right')

# Add the metric score values above each bar
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.2f}', xy=(rect.get_x() + rect.get_width() / 2, height), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom')

autolabel(rects1)
autolabel(rects2)
autolabel(rects3)
autolabel(rects4)

plt.tight_layout()

# Display the plot
plt.show()

In [ ]:
# Receiver Operating Characteristic Curve'
fig, ax = plt.subplots(figsize=(10, 6))

for classifier in classifiers:
    classifier.fit(scaled_x_train, y_train)
    y_pred = classifier.predict(scaled_x_val)
    
    roc_auc = roc_auc_score(y_val, classifier.predict(scaled_x_val))
    fpr, tpr, thresholds = roc_curve(y_val, classifier.predict_proba(scaled_x_val)[:,1])

    sns.set_style('darkgrid')
    ax.plot(fpr, tpr, label=f'{type(classifier).__name__} (area= {roc_auc:.2f})')


ax.plot([0, 1], [0, 1],'r--')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Receiver Operating Characteristic Curve')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

<a id="7"></a>
# <p style="background-image: url(https://i.postimg.cc/K87ByXmr/stage5.jpg);font-family:camtasia;font-size:75%;color:white;text-align:center;border-radius:15px 50px; padding:7px">Thank you for taking the time to review my notebook. If you have any questions or criticisms, please kindly let me know in the comments section.  </p>
